# NB06 — Musanga GEE verification: circles 13, 16, 21, 33

Prepare UAV RGB clipped to the four Nadine circles, and polygon/circle tables for an Earth Engine verification app.

**Run the QA cells first.** This notebook never changes the source shapefiles. Image export is local to Drive; uploading large raster images to GEE is a separate step. Earth Engine table exports require authentication and project permission. Public access must be granted explicitly in the Earth Engine Asset Manager after upload.

Inputs: `Final/Data/Annotations/Verification/Verification.shp`, `Circles_ROI_NAdine.shp`, four site RGB mosaics. Stable IDs are sorted by source feature order, and polygons are assigned to circle by centroid. Review counts/CRS and circle mapping before exporting.


In [ ]:
%pip -q install geopandas pyogrio rasterio earthengine-api geemap
from pathlib import Path
import json, re, math
import geopandas as gpd, numpy as np, pandas as pd, rasterio
from rasterio.windows import from_bounds, Window, transform as window_transform
from rasterio.mask import mask
from shapely.geometry import mapping
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
ROOT=Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final')
V=ROOT/'Data/Annotations/Verification/Verification.shp'
CANDIDATES=list((ROOT/'Data').rglob('Circles_ROI_NAdine.shp'))
RGB_DIR=ROOT/'Data/Ortho_RGB'
OUT=ROOT/'Outputs/V3/06_GEE_Verification'
OUT.mkdir(parents=True,exist_ok=True)
CIRCLE_IDS=[13,16,21,33]
DISPLAY_BUFFER_M=8
assert V.exists(),V
assert len(CANDIDATES)==1,('Expected unique circle shapefile',CANDIDATES)
CIRCLES_FILE=CANDIDATES[0]
print('Verification:',V)
print('Circle file:',CIRCLES_FILE)
print('RGB files:',[p.name for p in RGB_DIR.glob('*.tif')])
v=gpd.read_file(V)
circles=gpd.read_file(CIRCLES_FILE)
print('Verification fields:',list(v.columns),'count',len(v),'CRS',v.crs)
print('Circle fields:',list(circles.columns),'count',len(circles),'CRS',circles.crs)
assert v.crs is not None and circles.crs is not None
assert max(CIRCLE_IDS)<len(circles),'Circle IDs assumed zero-based source row numbers; verify indexing'
circles=circles.iloc[CIRCLE_IDS].copy()
circles['circle_id']=CIRCLE_IDS
circles=circles.to_crs(v.crs)
print('Chosen circle source rows:',CIRCLE_IDS)
display(circles[['circle_id']])


In [ ]:
from shapely.geometry import box
# IMPORTANT: resolve site mapping by spatial overlap; no arbitrary circle-to-site assumptions.
RGB_FILES=sorted(p for p in RGB_DIR.glob('*.tif') if not any(t in p.stem.lower() for t in ['dsm','ms_']))
assert RGB_FILES,'No RGB source rasters found'
site_rgb={}
for cid,cgeom in zip(circles.circle_id,circles.geometry):
    hits=[]
    for p in RGB_FILES:
        with rasterio.open(p) as ds:
            geom=gpd.GeoSeries([cgeom],crs=circles.crs).to_crs(ds.crs).iloc[0]
            coverage=geom.intersection(box(*ds.bounds)).area/max(geom.area,1e-9)
            if coverage>.95:hits.append((p,coverage))
    assert len(hits)==1,(cid,[(p.name,round(f,3)) for p,f in hits])
    site_rgb[int(cid)]=hits[0][0]
print('Circle-to-RGB mapping:')
for cid,p in site_rgb.items():print(cid,'->',p.name)
# Stable source-row polygon ID; centroid must lie within exactly one selected circle.
vv=v.to_crs(circles.crs).copy()
assigned=[]
for i,g in enumerate(vv.geometry):
    hits=[int(row.circle_id) for row in circles.itertuples() if row.geometry.covers(g.representative_point())]
    assigned.append(hits[0] if len(hits)==1 else None)
vv['circle_id']=assigned
selected=vv[vv.circle_id.notna()].copy()
selected['circle_id']=selected.circle_id.astype(int)
selected['verification_id']=[
    f'C{cid:02d}_V{int(i):05d}' for i,cid in zip(selected.index,selected.circle_id)
]
assert selected.verification_id.is_unique
print('Polygon counts by circle (assigned by representative point):')
display(selected.groupby('circle_id').size().rename('n_polygons').to_frame())
print('Polygons not assigned to these four circles:',len(vv)-len(selected))
# Check crossing circle boundaries; split/clip only after reviewing any exceptions.
for row in selected.itertuples():
    c=circles.loc[circles.circle_id==row.circle_id].geometry.iloc[0]
    fraction=row.geometry.intersection(c).area/max(row.geometry.area,1e-9)
    if fraction<0.99:print('BOUNDARY WARNING',row.verification_id,'inside fraction',round(fraction,3))
selected.to_file(OUT/'verification_4circles.geojson',driver='GeoJSON')
circles.to_file(OUT/'circles_4.geojson',driver='GeoJSON')
selected.drop(columns='geometry').to_csv(OUT/'verification_index.csv',index=False)
print('Saved two GeoJSONs and polygon-index CSV to',OUT)


In [ ]:
RUN_RGB_EXPORT=False # Set True only after inspecting circle-to-image mapping + polygon counts
if RUN_RGB_EXPORT:
    for row in circles.itertuples():
        cid=int(row.circle_id); rgb=site_rgb[cid]
        target=OUT/f'circle_{cid:02d}_UAV_RGB.tif'
        if target.exists():
            print('EXISTS:',target)
            continue
        with rasterio.open(rgb) as src:
            poly=gpd.GeoSeries([row.geometry.buffer(DISPLAY_BUFFER_M)],crs=circles.crs).to_crs(src.crs).iloc[0]
            data,tf=mask(src,[mapping(poly)],crop=True,filled=True,
                         nodata=src.nodata if src.nodata is not None else 0,indexes=[1,2,3])
            profile=src.profile.copy()
            profile.update(driver='GTiff',height=data.shape[1],width=data.shape[2],
                           transform=tf,count=3,compress='DEFLATE',tiled=True,
                           BIGTIFF='IF_SAFER')
            with rasterio.open(target,'w',**profile) as dst:dst.write(data)
        print('SAVED',target,'GB',round(target.stat().st_size/1e9,3),flush=True)
else:
    print('Preview only. Set RUN_RGB_EXPORT=True to write four clipped UAV GeoTIFFs.')


## Uploading assets to Earth Engine

First upload the four generated `circle_XX_UAV_RGB.tif` rasters as **image assets** using the Earth Engine Asset Manager (or `earthengine upload image` with Google Cloud Storage staging). Upload `verification_4circles.geojson` and `circles_4.geojson` as **table assets**, after verifying geometry and circle IDs.

Open [Earth Engine Code Editor](https://code.earthengine.google.com/) to create the app script and replace the asset IDs with the real image and feature collection paths. Grant public read access to imagery and vectors after upload, and then publish the app. GEE cannot persist browser-session classifications automatically; download the CSV before closing the app.
